
# 06 — Master Dataset Build  
## Final 5-Variable Structural Snapshot

This notebook builds the master country-year panel and the final 5-variable structural snapshots used for POSet analysis.

It runs after:

```text
05_Volatility_Features_2002_ContextOnly.ipynb
```

## Final decisions reflected here

- The full data acquisition / diagnostic panel starts from **2002**.
- The final POSet ordering set has **5 variables**, not 6.
- WGI/governance is retained only as contextual/sensitivity information.
- GDP volatility/stability is retained only as diagnostic/sensitivity information.
- GDP recovery is retained only as validation/outcome information.
- The POSet itself is built on **two single pre-shock cross-sections**:
  - 2007 for the Global Financial Crisis shock
  - 2019 for the COVID shock

## Final 5 ordering variables

The final structural ordering variables are:

| Raw variable | Final score | Direction |
|---|---|---|
| `public_debt_gdp_canonical` | `debt_capacity_score_0_100` | lower raw debt = better |
| `unemployment_rate` | `employment_strength_score_0_100` | lower unemployment = better |
| `rd_gdp` | `rd_intensity_score_0_100` | higher R&D/GDP = better |
| `tertiary_education_25_64` | `human_capital_tertiary_score_0_100` | higher tertiary attainment = better |
| `energy_import_dependency_raw` | `energy_security_score_0_100` | lower import dependency = better |

## Why the full panel is longer than the POSet snapshots

The 2002–2024/2025 panel is needed for:

- data acquisition and coverage diagnostics;
- GDP recovery construction;
- pre-shock volatility/context diagnostics;
- validation and interpretation;
- transparency about temporal scope.

The POSet itself uses only two representative pre-shock baseline years because it compares **structural capacity immediately before each major shock**, not long-run averages or post-shock outcomes.

## Main outputs

- `master_country_year_panel.csv`
- `master_country_year_panel_data_dictionary.csv`
- `structural_snapshot_final5_2007_2019.csv`
- `structural_snapshot_final5_complete_cases.csv`
- `structural_snapshot_final5_score_parameters.csv`
- `master_structural_snapshot_coverage.csv`
- `master_problem_cases.csv`
- `report_ready_scope_and_measure_note.csv`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

COMPARABLE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files"
RECOVERY_DIR = PROJECT_ROOT / "Data" / "Processed" / "GDP_Recovery_Dynamic_Baseline"
WGI_DIR = PROJECT_ROOT / "Data" / "Processed" / "WGI_Governance_ContextOnly"
VOLATILITY_DIR = PROJECT_ROOT / "Data" / "Processed" / "Volatility_Features"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "06_Master_Dataset_Build"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Master_Dataset"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

START_YEAR = 2002
END_YEAR = 2025

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Comparable folder:", COMPARABLE_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_190808
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Comparable folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Master_Dataset


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

STANDARDIZED_LONG_CANDIDATES = [
    COMPARABLE_DIR / "standardized_country_year_variable_long.csv",
    COMPARABLE_DIR / "standardized_all_variables_long.csv",
]

RECOVERY_SUMMARY_CANDIDATES = [
    RECOVERY_DIR / "country_recovery_summary_dynamic_baseline_2007_2019.csv",
    PROJECT_ROOT / "Data" / "Diagnostics" / "03_GDP_Recovery_Dynamic_Baseline" / "country_recovery_summary_dynamic_baseline_2007_2019.csv",
]

WGI_CONTEXT_CANDIDATES = [
    WGI_DIR / "wgi_governance_context_country_year.csv",
    WGI_DIR / "wgi_final_project_ready_country_year_2002_2024.csv",
]

VOLATILITY_FEATURE_CANDIDATES = [
    VOLATILITY_DIR / "gdp_growth_stability_features_country_shock.csv",
]

def find_first_existing_path(candidates, label, required=True):
    for path in candidates:
        if path.exists():
            print(f"Using {label}: {path}")
            return path
    if required:
        raise FileNotFoundError(
            f"Could not find {label}. Tried:\n"
            + "\n".join([f"- {p}" for p in candidates])
        )
    print(f"{label} not found. Continuing without it.")
    return None

STANDARDIZED_LONG_FILE = find_first_existing_path(
    STANDARDIZED_LONG_CANDIDATES,
    "standardized comparable long dataset",
)

RECOVERY_SUMMARY_FILE = find_first_existing_path(
    RECOVERY_SUMMARY_CANDIDATES,
    "GDP recovery summary",
    required=True,
)

WGI_CONTEXT_FILE = find_first_existing_path(
    WGI_CONTEXT_CANDIDATES,
    "WGI contextual dataset",
    required=False,
)

VOLATILITY_FEATURE_FILE = find_first_existing_path(
    VOLATILITY_FEATURE_CANDIDATES,
    "volatility feature dataset",
    required=False,
)

FINAL_5_RAW_TO_SCORE = {
    "public_debt_gdp_canonical": {
        "score_column": "debt_capacity_score_0_100",
        "concept": "Debt capacity",
        "direction": "lower_is_better",
        "report_note": "Lower public debt/GDP is interpreted as greater debt capacity.",
    },
    "unemployment_rate": {
        "score_column": "employment_strength_score_0_100",
        "concept": "Employment strength",
        "direction": "lower_is_better",
        "report_note": "Lower unemployment is interpreted as greater labour-market strength.",
    },
    "rd_gdp": {
        "score_column": "rd_intensity_score_0_100",
        "concept": "R&D intensity",
        "direction": "higher_is_better",
        "report_note": "Higher R&D expenditure as a percentage of GDP is interpreted as stronger innovation capacity.",
    },
    "tertiary_education_25_64": {
        "score_column": "human_capital_tertiary_score_0_100",
        "concept": "Human capital",
        "direction": "higher_is_better",
        "report_note": "Higher adult tertiary educational attainment, age 25–64, is interpreted as stronger human-capital stock.",
    },
    "energy_import_dependency_raw": {
        "score_column": "energy_security_score_0_100",
        "concept": "Energy security",
        "direction": "lower_is_better",
        "report_note": "Lower net energy import dependency is interpreted as greater energy security. Negative raw values can occur for net exporters and are not clipped.",
    },
}

FINAL_5_RAW_VARIABLES = list(FINAL_5_RAW_TO_SCORE.keys())
FINAL_5_SCORE_COLUMNS = [v["score_column"] for v in FINAL_5_RAW_TO_SCORE.values()]

VALIDATION_VARIABLES = [
    "gdp_growth",
    "unemployment_rate",
    "inflation_cpi",
    "public_debt_gdp_canonical",
    "productivity_gdp_per_hour",
]

BASELINE_CONFIGS = [
    {
        "shock_id": "2007",
        "shock_label": "Global Financial Crisis",
        "baseline_year": 2007,
        "recovery_column": "Recovery_2007",
        "recovery_status_column": "Recovery_2007_status",
    },
    {
        "shock_id": "2019",
        "shock_label": "COVID shock",
        "baseline_year": 2019,
        "recovery_column": "Recovery_2019",
        "recovery_status_column": "Recovery_2019_status",
    },
]

print("Final 5 score columns:")
for c in FINAL_5_SCORE_COLUMNS:
    print("-", c)


Using standardized comparable long dataset: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files\standardized_country_year_variable_long.csv
Using GDP recovery summary: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\GDP_Recovery_Dynamic_Baseline\country_recovery_summary_dynamic_baseline_2007_2019.csv
Using WGI contextual dataset: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\WGI_Governance_ContextOnly\wgi_governance_context_country_year.csv
Using volatility feature dataset: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Volatility_Features\gdp_growth_stability_features_country_shock.csv
Final 5 score columns:
- debt_capacity

In [3]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


In [4]:

# ------------------------------------------------------
# LOAD STANDARDIZED LONG DATA AND PIVOT TO MASTER PANEL
# ------------------------------------------------------

long_df = pd.read_csv(STANDARDIZED_LONG_FILE)

rename_candidates = {
    "country": "Country",
    "iso3": "Country",
    "ISO3": "Country",
    "year": "Year",
    "Variable": "variable",
    "variable_name": "variable",
    "value": "Value",
    "obs_value": "Value",
    "OBS_VALUE": "Value",
}

for old_col, new_col in rename_candidates.items():
    if old_col in long_df.columns and new_col not in long_df.columns:
        long_df = long_df.rename(columns={old_col: new_col})

required_cols = {"Country", "Year", "variable", "Value"}
missing_cols = required_cols - set(long_df.columns)

if missing_cols:
    raise ValueError(
        f"Standardized long file missing required columns: {missing_cols}\n"
        f"Selected file: {STANDARDIZED_LONG_FILE}\n"
        f"Available columns: {list(long_df.columns)}"
    )

long_df["Country"] = long_df["Country"].astype(str).str.strip().str.upper()
long_df["Year"] = pd.to_numeric(long_df["Year"], errors="coerce")
long_df["Value"] = pd.to_numeric(long_df["Value"], errors="coerce")
long_df["variable"] = long_df["variable"].astype(str).str.strip()

long_df = (
    long_df.dropna(subset=["Country", "Year", "variable", "Value"])
    .assign(Year=lambda d: d["Year"].astype(int))
    .query("Year >= @START_YEAR and Year <= @END_YEAR")
    .copy()
)

# Keep only ISO3-style standardized country codes for the master panel.
invalid_country_codes = long_df[~long_df["Country"].str.match(r"^[A-Z]{3}$", na=False)].copy()
invalid_country_codes.to_csv(DIAGNOSTICS_DIR / "invalid_country_codes_in_master_input.csv", index=False)

if len(invalid_country_codes) > 0:
    raise ValueError("Invalid country codes found in master input. Check invalid_country_codes_in_master_input.csv.")

missing_final5 = sorted(set(FINAL_5_RAW_VARIABLES) - set(long_df["variable"].unique()))
if missing_final5:
    raise ValueError(f"Missing final 5 raw variables in standardized long data: {missing_final5}")

master_country_year_panel = (
    long_df
    .pivot_table(
        index=["Country", "Year"],
        columns="variable",
        values="Value",
        aggfunc="first",
    )
    .reset_index()
)

master_country_year_panel.columns.name = None
master_country_year_panel = master_country_year_panel.sort_values(["Country", "Year"]).reset_index(drop=True)

print("Master panel shape:", master_country_year_panel.shape)
print("Countries:", master_country_year_panel["Country"].nunique())
print("Years:", master_country_year_panel["Year"].min(), "-", master_country_year_panel["Year"].max())
display(master_country_year_panel.head())


Master panel shape: (3499, 12)
Countries: 177
Years: 2002 - 2025


,Country,Year,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,public_debt_gdp_eurostat,public_debt_gdp_worldbank,rd_gdp,tertiary_education_25_64,unemployment_rate
0,AGO,2002,-575.4543,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AGO,2003,-520.1810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AGO,2004,-590.3533,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AGO,2005,-812.8839,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AGO,2006,-818.5898,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:

# ------------------------------------------------------
# MERGE RECOVERY OUTCOMES
# ------------------------------------------------------

recovery = pd.read_csv(RECOVERY_SUMMARY_FILE)
recovery["Country"] = recovery["Country"].astype(str).str.strip().str.upper()

required_recovery_cols = {"Country", "Recovery_2007", "Recovery_2019"}
missing_recovery_cols = required_recovery_cols - set(recovery.columns)

if missing_recovery_cols:
    raise ValueError(f"Recovery file missing required columns: {missing_recovery_cols}")

master_country_year_panel = master_country_year_panel.merge(
    recovery,
    on="Country",
    how="left",
)

print("Merged recovery outcomes.")
display(master_country_year_panel[["Country", "Year", "Recovery_2007", "Recovery_2019"]].head())


Merged recovery outcomes.


,Country,Year,Recovery_2007,Recovery_2019
0,AGO,2002,NaN,NaN
1,AGO,2003,NaN,NaN
2,AGO,2004,NaN,NaN
3,AGO,2005,NaN,NaN
4,AGO,2006,NaN,NaN


In [6]:

# ------------------------------------------------------
# MERGE WGI CONTEXTUAL DATA IF AVAILABLE
# ------------------------------------------------------
# WGI is context/sensitivity only, not part of final 5 ordering.

if WGI_CONTEXT_FILE is not None and WGI_CONTEXT_FILE.exists():
    wgi_context = pd.read_csv(WGI_CONTEXT_FILE)
    wgi_context["Country"] = wgi_context["Country"].astype(str).str.strip().str.upper()
    wgi_context["Year"] = pd.to_numeric(wgi_context["Year"], errors="coerce").astype("Int64")

    # Keep only useful WGI columns.
    wgi_cols_keep = [
        "Country", "Year",
        "wgi_va_score", "wgi_pv_score", "wgi_ge_score",
        "wgi_rq_score", "wgi_rl_score", "wgi_cc_score",
        "wgi_six_dimension_mean_score",
        "wgi_six_dimension_available_count",
        "governance_context_score_0_100",
        "governance_context_role",
    ]
    wgi_cols_keep = [c for c in wgi_cols_keep if c in wgi_context.columns]

    wgi_context = wgi_context[wgi_cols_keep].copy()
    wgi_context["Year"] = wgi_context["Year"].astype(int)

    master_country_year_panel = master_country_year_panel.merge(
        wgi_context,
        on=["Country", "Year"],
        how="left",
    )

    print("Merged WGI context columns:", [c for c in wgi_cols_keep if c not in {"Country", "Year"}])

else:
    wgi_context = pd.DataFrame()
    print("WGI context file not found. Continuing without WGI context.")

display(master_country_year_panel.head())


Merged WGI context columns: ['wgi_va_score', 'wgi_pv_score', 'wgi_ge_score', 'wgi_rq_score', 'wgi_rl_score', 'wgi_cc_score', 'wgi_six_dimension_mean_score', 'wgi_six_dimension_available_count', 'governance_context_score_0_100', 'governance_context_role']


,Country,Year,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,public_debt_gdp_eurostat,public_debt_gdp_worldbank,rd_gdp,tertiary_education_25_64,unemployment_rate,Recovery_2007,Recovery_2019,Recovery_2007_status,Recovery_2019_status,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score,wgi_six_dimension_mean_score,wgi_six_dimension_available_count,governance_context_score_0_100,governance_context_role
0,AGO,2002,-575.4543,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.3847,-1.8158,-1.2301,-1.6064,-1.4962,-1.3041,-1.4729,6.0000,16.9845,contextual_or_sensitivity_only_not_final_ordering
1,AGO,2003,-520.1810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.3097,-1.2507,-1.4582,-1.6438,-1.6751,-1.3231,-1.4435,6.0000,17.6813,contextual_or_sensitivity_only_not_final_ordering
2,AGO,2004,-590.3533,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.3167,-1.1591,-1.5317,-1.4291,-1.4539,-1.2229,-1.3522,6.0000,19.8408,contextual_or_sensitivity_only_not_final_ordering
3,AGO,2005,-812.8839,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.2088,-0.9718,-1.5412,-1.4092,-1.3592,-1.2941,-1.2974,6.0000,21.1394,contextual_or_sensitivity_only_not_final_ordering
4,AGO,2006,-818.5898,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.2632,-0.6714,-1.4925,-1.0973,-1.3263,-1.2968,-1.1913,6.0000,23.6518,contextual_or_sensitivity_only_not_final_ordering


In [7]:

# ------------------------------------------------------
# SAVE MASTER COUNTRY-YEAR PANEL
# ------------------------------------------------------

save_table(
    master_country_year_panel,
    "master_country_year_panel.csv",
    "Master country-year panel",
    "Wide country-year panel with standardized raw variables, recovery outcomes, and contextual WGI fields where available.",
)

master_variable_coverage_summary = (
    master_country_year_panel
    .drop(columns=["Country", "Year"])
    .notna()
    .sum()
    .reset_index()
    .rename(columns={"index": "variable", 0: "non_missing_rows"})
)

master_variable_coverage_summary["countries_with_data"] = [
    master_country_year_panel.loc[master_country_year_panel[col].notna(), "Country"].nunique()
    for col in master_variable_coverage_summary["variable"]
]
master_variable_coverage_summary["min_year"] = [
    master_country_year_panel.loc[master_country_year_panel[col].notna(), "Year"].min()
    for col in master_variable_coverage_summary["variable"]
]
master_variable_coverage_summary["max_year"] = [
    master_country_year_panel.loc[master_country_year_panel[col].notna(), "Year"].max()
    for col in master_variable_coverage_summary["variable"]
]

save_table(
    master_variable_coverage_summary,
    "master_variable_coverage_summary.csv",
    "Master variable coverage summary",
    "Coverage summary for variables in the master country-year panel.",
)

display(master_variable_coverage_summary.head(50))


Saved: master_country_year_panel.csv
Saved: master_variable_coverage_summary.csv


,variable,non_missing_rows,countries_with_data,min_year,max_year
0,energy_import_dependency_raw,3038,153,2002,2023
1,gdp_growth,1043,44,2002,2025
2,inflation_cpi,1116,48,2002,2025
3,productivity_gdp_per_hour,923,41,2002,2024
4,public_debt_gdp_canonical,1662,108,2002,2025
5,public_debt_gdp_eurostat,648,27,2002,2025
6,public_debt_gdp_worldbank,1060,83,2002,2024
7,rd_gdp,998,47,2002,2025
8,tertiary_education_25_64,915,48,2002,2024
9,unemployment_rate,1194,53,2002,2025


In [8]:

# ------------------------------------------------------
# BUILD STRUCTURAL SNAPSHOTS FOR 2007 AND 2019
# ------------------------------------------------------

snapshot_frames = []

for cfg in BASELINE_CONFIGS:
    baseline_year = cfg["baseline_year"]
    shock_id = cfg["shock_id"]
    shock_label = cfg["shock_label"]

    snapshot = master_country_year_panel[
        master_country_year_panel["Year"].eq(baseline_year)
    ].copy()

    snapshot["shock_id"] = shock_id
    snapshot["shock_label"] = shock_label
    snapshot["baseline_year"] = baseline_year
    snapshot["poset_temporal_design"] = "single_pre_shock_cross_section"

    raw_cols = FINAL_5_RAW_VARIABLES

    snapshot["final5_raw_variables_available"] = snapshot[raw_cols].notna().sum(axis=1)
    snapshot["complete_case_final5_raw"] = snapshot["final5_raw_variables_available"].eq(len(raw_cols))

    snapshot_frames.append(snapshot)

structural_snapshot_final5_2007_2019 = pd.concat(snapshot_frames, ignore_index=True)

# Merge volatility features by Country + shock_id if available.
if VOLATILITY_FEATURE_FILE is not None and VOLATILITY_FEATURE_FILE.exists():
    volatility_features = pd.read_csv(VOLATILITY_FEATURE_FILE)
    volatility_features["Country"] = volatility_features["Country"].astype(str).str.strip().str.upper()
    volatility_features["shock_id"] = volatility_features["shock_id"].astype(str)

    vol_cols = [
        "Country", "shock_id",
        "gdp_growth_volatility_sd",
        "gdp_growth_stability_score_0_100",
        "years_available",
        "window_start",
        "window_end",
        "sufficient_window_coverage",
        "volatility_feature_role",
    ]
    vol_cols = [c for c in vol_cols if c in volatility_features.columns]

    structural_snapshot_final5_2007_2019 = structural_snapshot_final5_2007_2019.merge(
        volatility_features[vol_cols],
        on=["Country", "shock_id"],
        how="left",
        suffixes=("", "_volatility"),
    )
else:
    volatility_features = pd.DataFrame()

print("Structural snapshot rows:", len(structural_snapshot_final5_2007_2019))
display(structural_snapshot_final5_2007_2019.head())


Structural snapshot rows: 313


,Country,Year,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,public_debt_gdp_eurostat,public_debt_gdp_worldbank,rd_gdp,tertiary_education_25_64,unemployment_rate,Recovery_2007,Recovery_2019,Recovery_2007_status,Recovery_2019_status,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score,wgi_six_dimension_mean_score,wgi_six_dimension_available_count,governance_context_score_0_100,governance_context_role,shock_id,shock_label,baseline_year,poset_temporal_design,final5_raw_variables_available,complete_case_final5_raw,gdp_growth_volatility_sd,gdp_growth_stability_score_0_100,years_available,window_start,window_end,sufficient_window_coverage,volatility_feature_role
0,AGO,2007,-905.0514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.2352,-0.7471,-1.4130,-1.0725,-1.3310,-1.2804,-1.1799,6.0000,23.9216,contextual_or_sensitivity_only_not_final_ordering,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ALB,2007,49.3499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0981,-0.1886,-0.5172,-0.0069,-0.5254,-0.6752,-0.3025,6.0000,44.6916,contextual_or_sensitivity_only_not_final_ordering,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ARB,2007,-202.6365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ARE,2007,-239.1435,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.6165,1.0199,0.9034,0.7403,0.0361,0.7780,0.4769,6.0000,63.1436,contextual_or_sensitivity_only_not_final_ordering,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ARG,2007,-8.7584,NaN,NaN,NaN,NaN,NaN,NaN,0.4601,NaN,NaN,NaN,NaN,NaN,NaN,0.5041,0.2041,0.0009,-0.6472,-0.4645,-0.4043,-0.1345,6.0000,48.6699,contextual_or_sensitivity_only_not_final_ordering,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,2,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:

# ------------------------------------------------------
# CREATE DIRECTION-ALIGNED 0–100 SCORES FOR FINAL 5 VARIABLES
# ------------------------------------------------------
# Scores are calculated separately within each baseline cross-section
# using complete final-5 raw cases for that shock.
#
# These scores are not a scalar index. They only direction-align variables
# before ordinal profiling / POSet construction.

score_parameter_rows = []
scored_snapshot_frames = []

for shock_id, snapshot in structural_snapshot_final5_2007_2019.groupby("shock_id"):
    snapshot = snapshot.copy()
    complete = snapshot[snapshot["complete_case_final5_raw"]].copy()

    if complete.empty:
        raise ValueError(f"No complete final-5 cases for shock {shock_id}.")

    for raw_col, meta in FINAL_5_RAW_TO_SCORE.items():
        score_col = meta["score_column"]
        direction = meta["direction"]

        min_value = complete[raw_col].min()
        max_value = complete[raw_col].max()

        if pd.isna(min_value) or pd.isna(max_value) or min_value == max_value:
            snapshot[score_col] = np.nan
            score_note = "Could not score because min/max unavailable or equal."
        else:
            if direction == "higher_is_better":
                snapshot[score_col] = (snapshot[raw_col] - min_value) / (max_value - min_value) * 100
            elif direction == "lower_is_better":
                snapshot[score_col] = (max_value - snapshot[raw_col]) / (max_value - min_value) * 100
            else:
                raise ValueError(f"Unknown direction: {direction}")

            score_note = "Min-max direction-aligned within complete-case baseline cross-section."

        score_parameter_rows.append({
            "shock_id": shock_id,
            "baseline_year": int(snapshot["baseline_year"].iloc[0]),
            "raw_variable": raw_col,
            "score_column": score_col,
            "concept": meta["concept"],
            "direction": direction,
            "min_value_complete_cases": min_value,
            "max_value_complete_cases": max_value,
            "scoring_note": score_note,
            "is_final_ordering_variable": True,
        })

    snapshot["final5_score_variables_available"] = snapshot[FINAL_5_SCORE_COLUMNS].notna().sum(axis=1)
    snapshot["complete_case_final5_scores"] = snapshot["final5_score_variables_available"].eq(len(FINAL_5_SCORE_COLUMNS))

    scored_snapshot_frames.append(snapshot)

structural_snapshot_final5_2007_2019 = pd.concat(scored_snapshot_frames, ignore_index=True)

structural_snapshot_final5_complete_cases = structural_snapshot_final5_2007_2019[
    structural_snapshot_final5_2007_2019["complete_case_final5_scores"]
].copy()

structural_snapshot_final5_score_parameters = pd.DataFrame(score_parameter_rows)

final5_ordering_variable_score_map = pd.DataFrame([
    {
        "raw_variable": raw_col,
        "score_column": meta["score_column"],
        "concept": meta["concept"],
        "direction": meta["direction"],
        "report_note": meta["report_note"],
        "is_final_ordering_variable": True,
    }
    for raw_col, meta in FINAL_5_RAW_TO_SCORE.items()
])

save_table(
    structural_snapshot_final5_2007_2019,
    "structural_snapshot_final5_2007_2019.csv",
    "Final 5 structural snapshot 2007 and 2019",
    "Country-shock structural snapshot with raw variables, direction-aligned scores, recovery, WGI context, and volatility diagnostics.",
)

save_table(
    structural_snapshot_final5_complete_cases,
    "structural_snapshot_final5_complete_cases.csv",
    "Final 5 complete-case structural snapshot",
    "Complete-case countries for final 5 score variables in each POSet baseline cross-section.",
)

save_table(
    structural_snapshot_final5_score_parameters,
    "structural_snapshot_final5_score_parameters.csv",
    "Final 5 score parameters",
    "Shock-specific min/max score parameters used to direction-align the five final ordering variables.",
)

save_table(
    final5_ordering_variable_score_map,
    "final5_ordering_variable_score_map.csv",
    "Final 5 ordering variable score map",
    "Mapping from raw variables to direction-aligned score columns.",
)

display(structural_snapshot_final5_score_parameters)
display(structural_snapshot_final5_complete_cases.head())


Saved: structural_snapshot_final5_2007_2019.csv
Saved: structural_snapshot_final5_complete_cases.csv
Saved: structural_snapshot_final5_score_parameters.csv
Saved: final5_ordering_variable_score_map.csv


,shock_id,baseline_year,raw_variable,score_column,concept,direction,min_value_complete_cases,max_value_complete_cases,scoring_note,is_final_ordering_variable
0,2007,2007,public_debt_gdp_canonical,debt_capacity_score_0_100,Debt capacity,lower_is_better,3.9000,104.6000,Min-max direction-aligned within complete-case...,True
1,2007,2007,unemployment_rate,employment_strength_score_0_100,Employment strength,lower_is_better,3.3760,11.0170,Min-max direction-aligned within complete-case...,True
2,2007,2007,rd_gdp,rd_intensity_score_0_100,R&D intensity,higher_is_better,0.4473,3.3373,Min-max direction-aligned within complete-case...,True
3,2007,2007,tertiary_education_25_64,human_capital_tertiary_score_0_100,Human capital,higher_is_better,13.5781,48.1816,Min-max direction-aligned within complete-case...,True
4,2007,2007,energy_import_dependency_raw,energy_security_score_0_100,Energy security,lower_is_better,-54.5900,106.6438,Min-max direction-aligned within complete-case...,True
5,2019,2019,public_debt_gdp_canonical,debt_capacity_score_0_100,Debt capacity,lower_is_better,9.0000,183.2000,Min-max direction-aligned within complete-case...,True
6,2019,2019,unemployment_rate,employment_strength_score_0_100,Employment strength,lower_is_better,2.0140,28.4460,Min-max direction-aligned within complete-case...,True
7,2019,2019,rd_gdp,rd_intensity_score_0_100,R&D intensity,higher_is_better,0.3220,4.3638,Min-max direction-aligned within complete-case...,True
8,2019,2019,tertiary_education_25_64,human_capital_tertiary_score_0_100,Human capital,higher_is_better,15.7982,59.0203,Min-max direction-aligned within complete-case...,True
9,2019,2019,energy_import_dependency_raw,energy_security_score_0_100,Energy security,lower_is_better,-230.5402,109.5148,Min-max direction-aligned within complete-case...,True


,Country,Year,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,public_debt_gdp_eurostat,public_debt_gdp_worldbank,rd_gdp,tertiary_education_25_64,unemployment_rate,Recovery_2007,Recovery_2019,Recovery_2007_status,Recovery_2019_status,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score,wgi_six_dimension_mean_score,wgi_six_dimension_available_count,governance_context_score_0_100,governance_context_role,shock_id,shock_label,baseline_year,poset_temporal_design,final5_raw_variables_available,complete_case_final5_raw,gdp_growth_volatility_sd,gdp_growth_stability_score_0_100,years_available,window_start,window_end,sufficient_window_coverage,volatility_feature_role,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,final5_score_variables_available,complete_case_final5_scores
7,AUT,2007,70.1726,3.7752,2.1686,76.8401,65.8000,65.8000,NaN,2.4336,25.0927,4.8580,2.0000,2.0000,recovered,recovered,1.6047,1.3180,1.7742,1.8293,1.9455,1.9654,1.7395,6.0000,93.0358,contextual_or_sensitivity_only_not_final_ordering,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,5,True,1.0098,80.5768,6.0000,2002.0000,2007.0000,True,diagnostic_or_sensitivity_only_not_final_ordering,38.5303,80.6046,68.7303,33.2757,22.6201,5,True
9,BEL,2007,91.1428,3.6769,1.8231,84.6487,87.5000,87.5000,NaN,1.8500,32.0949,7.4580,1.0000,1.0000,recovered,recovered,1.5790,0.8537,1.5125,1.4649,1.3583,1.3867,1.3592,6.0000,84.0309,contextual_or_sensitivity_only_not_final_ordering,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,5,True,1.0326,79.7510,6.0000,2002.0000,2007.0000,True,diagnostic_or_sensitivity_only_not_final_ordering,16.9811,46.5777,48.5360,53.5113,9.6139,5,True
24,CAN,2007,-54.5900,2.0499,2.1384,53.5208,39.1586,NaN,39.1586,1.9036,48.1816,6.1520,1.0000,1.0000,recovered,recovered,1.5813,1.0423,1.8205,1.6475,1.7329,1.9345,1.6265,6.0000,90.3599,contextual_or_sensitivity_only_not_final_ordering,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,5,True,0.5834,96.0628,6.0000,2002.0000,2007.0000,True,diagnostic_or_sensitivity_only_not_final_ordering,64.9865,63.6697,50.3901,100.0000,100.0000,5,True
37,CZE,2007,25.1301,5.4888,2.8531,44.8732,27.3000,27.3000,NaN,1.2967,13.7327,5.3160,5.0000,2.0000,recovered,recovered,0.9561,1.0971,0.6145,1.1204,0.7261,0.4279,0.8237,6.0000,71.3538,contextual_or_sensitivity_only_not_final_ordering,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,5,True,1.9614,46.0165,6.0000,2002.0000,2007.0000,True,diagnostic_or_sensitivity_only_not_final_ordering,76.7627,74.6107,29.3904,0.4466,50.5562,5,True
38,DEU,2007,60.5578,2.8891,2.2983,75.0604,63.7000,63.7000,NaN,2.4192,24.3000,8.6370,2.0000,2.0000,recovered,recovered,1.5753,1.0605,1.7597,1.5280,1.7388,1.6554,1.5530,6.0000,88.6192,contextual_or_sensitivity_only_not_final_ordering,2007,Global Financial Crisis,2007,single_pre_shock_cross_section,5,True,1.7308,54.3926,6.0000,2002.0000,2007.0000,True,diagnostic_or_sensitivity_only_not_final_ordering,40.6157,31.1478,68.2315,30.9850,28.5833,5,True


In [10]:

# ------------------------------------------------------
# STRUCTURAL SNAPSHOT COVERAGE
# ------------------------------------------------------

master_structural_snapshot_coverage = (
    structural_snapshot_final5_2007_2019
    .groupby(["shock_id", "baseline_year"])
    .agg(
        countries_in_baseline_year=("Country", "nunique"),
        complete_final5_raw_countries=("complete_case_final5_raw", "sum"),
        complete_final5_score_countries=("complete_case_final5_scores", "sum"),
        mean_final5_raw_variables_available=("final5_raw_variables_available", "mean"),
        mean_final5_score_variables_available=("final5_score_variables_available", "mean"),
    )
    .reset_index()
)

score_range_summary = (
    structural_snapshot_final5_complete_cases
    .groupby("shock_id")[FINAL_5_SCORE_COLUMNS]
    .agg(["min", "median", "max"])
)

save_table(
    master_structural_snapshot_coverage,
    "master_structural_snapshot_coverage.csv",
    "Master structural snapshot coverage",
    "Coverage summary for 2007 and 2019 final 5 structural POSet snapshots.",
)

# Flatten score range summary.
score_range_summary_flat = score_range_summary.copy()
score_range_summary_flat.columns = ["_".join(col).strip() for col in score_range_summary_flat.columns.values]
score_range_summary_flat = score_range_summary_flat.reset_index()

save_table(
    score_range_summary_flat,
    "final5_score_range_summary.csv",
    "Final 5 score range summary",
    "Score min/median/max by shock for complete-case countries.",
)

display(master_structural_snapshot_coverage)
display(score_range_summary_flat)


Saved: master_structural_snapshot_coverage.csv
Saved: final5_score_range_summary.csv


,shock_id,baseline_year,countries_in_baseline_year,complete_final5_raw_countries,complete_final5_score_countries,mean_final5_raw_variables_available,mean_final5_score_variables_available
0,2007,2007,152,25,25,2.1974,2.1974
1,2019,2019,161,35,35,2.2857,2.2857


,shock_id,debt_capacity_score_0_100_min,debt_capacity_score_0_100_median,debt_capacity_score_0_100_max,employment_strength_score_0_100_min,employment_strength_score_0_100_median,employment_strength_score_0_100_max,rd_intensity_score_0_100_min,rd_intensity_score_0_100_median,rd_intensity_score_0_100_max,human_capital_tertiary_score_0_100_min,human_capital_tertiary_score_0_100_median,human_capital_tertiary_score_0_100_max,energy_security_score_0_100_min,energy_security_score_0_100_median,energy_security_score_0_100_max
0,2007,0.0000,64.8461,100.0000,0.0000,64.6774,100.0000,0.0000,34.0933,100.0000,0.0000,37.7650,100.0000,0.0000,28.5833,100.0000
1,2019,0.0000,73.1343,100.0000,0.0000,88.0864,100.0000,0.0000,28.2727,100.0000,0.0000,51.1465,100.0000,0.0000,16.6637,100.0000


In [11]:

# ------------------------------------------------------
# PROBLEM CASES AND SANITY CHECKS
# ------------------------------------------------------

problem_rows = []

# Negative public debt in baseline snapshots.
if "public_debt_gdp_canonical" in structural_snapshot_final5_2007_2019.columns:
    negative_debt = structural_snapshot_final5_2007_2019[
        structural_snapshot_final5_2007_2019["public_debt_gdp_canonical"].notna()
        & (structural_snapshot_final5_2007_2019["public_debt_gdp_canonical"] < 0)
    ].copy()

    for _, row in negative_debt.iterrows():
        problem_rows.append({
            "Country": row["Country"],
            "shock_id": row["shock_id"],
            "baseline_year": row["baseline_year"],
            "problem_type": "negative_public_debt_gdp_canonical",
            "variable": "public_debt_gdp_canonical",
            "value": row["public_debt_gdp_canonical"],
            "severity": "review",
            "note": "Negative public debt/GDP found in baseline snapshot. Do not automatically drop; review source context.",
        })

# Missing final 5 raw variables.
missing_final5_snapshot = structural_snapshot_final5_2007_2019[
    ~structural_snapshot_final5_2007_2019["complete_case_final5_raw"]
].copy()

for _, row in missing_final5_snapshot.iterrows():
    missing_vars = [v for v in FINAL_5_RAW_VARIABLES if pd.isna(row.get(v))]
    problem_rows.append({
        "Country": row["Country"],
        "shock_id": row["shock_id"],
        "baseline_year": row["baseline_year"],
        "problem_type": "incomplete_final5_baseline",
        "variable": ",".join(missing_vars),
        "value": np.nan,
        "severity": "exclude_from_main_poset_for_that_shock",
        "note": "Missing at least one final 5 ordering raw variable in this baseline year.",
    })

master_problem_cases = pd.DataFrame(problem_rows)

if master_problem_cases.empty:
    master_problem_cases = pd.DataFrame(columns=[
        "Country", "shock_id", "baseline_year", "problem_type",
        "variable", "value", "severity", "note",
    ])

save_table(
    master_problem_cases,
    "master_problem_cases.csv",
    "Master problem cases",
    "Baseline snapshot cases needing review or exclusion from main POSet complete-case sample.",
)

display(master_problem_cases.head(100))


Saved: master_problem_cases.csv


,Country,shock_id,baseline_year,problem_type,variable,value,severity,note
0,AGO,2007,2007,incomplete_final5_baseline,"public_debt_gdp_canonical,unemployment_rate,rd...",NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...
1,ALB,2007,2007,incomplete_final5_baseline,"public_debt_gdp_canonical,unemployment_rate,rd...",NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...
2,ARB,2007,2007,incomplete_final5_baseline,"public_debt_gdp_canonical,unemployment_rate,rd...",NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...
3,ARE,2007,2007,incomplete_final5_baseline,"public_debt_gdp_canonical,unemployment_rate,rd...",NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...
4,ARG,2007,2007,incomplete_final5_baseline,"public_debt_gdp_canonical,unemployment_rate,te...",NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...
5,ARM,2007,2007,incomplete_final5_baseline,"public_debt_gdp_canonical,unemployment_rate,rd...",NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...
6,AUS,2007,2007,incomplete_final5_baseline,rd_gdp,NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...
7,AZE,2007,2007,incomplete_final5_baseline,"public_debt_gdp_canonical,unemployment_rate,rd...",NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...
8,BEN,2007,2007,incomplete_final5_baseline,"public_debt_gdp_canonical,unemployment_rate,rd...",NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...
9,BGD,2007,2007,incomplete_final5_baseline,"public_debt_gdp_canonical,unemployment_rate,rd...",NaN,exclude_from_main_poset_for_that_shock,Missing at least one final 5 ordering raw vari...


In [12]:

# ------------------------------------------------------
# VOLATILITY LEAKAGE CHECK
# ------------------------------------------------------
# Volatility windows must end no later than the baseline year.
# They are diagnostic/sensitivity only and must not include post-shock recovery information.

if VOLATILITY_FEATURE_FILE is not None and VOLATILITY_FEATURE_FILE.exists():
    v = volatility_features.copy()

    # Bring baseline year mapping.
    baseline_map = pd.DataFrame(BASELINE_CONFIGS)[["shock_id", "baseline_year"]]
    baseline_map["shock_id"] = baseline_map["shock_id"].astype(str)

    v["shock_id"] = v["shock_id"].astype(str)
    v = v.merge(baseline_map, on="shock_id", how="left", suffixes=("", "_expected"))

    if "window_end" in v.columns:
        v["leakage_flag_window_after_baseline"] = v["window_end"] > v["baseline_year"]
    else:
        v["leakage_flag_window_after_baseline"] = np.nan

    snapshot_volatility_leakage_check = (
        v.groupby("shock_id")
        .agg(
            rows=("Country", "size"),
            countries=("Country", "nunique"),
            min_window_start=("window_start", "min") if "window_start" in v.columns else ("Country", "size"),
            max_window_end=("window_end", "max") if "window_end" in v.columns else ("Country", "size"),
            baseline_year=("baseline_year", "first"),
            leakage_flags=("leakage_flag_window_after_baseline", "sum"),
        )
        .reset_index()
    )
else:
    snapshot_volatility_leakage_check = pd.DataFrame([
        {
            "shock_id": "not_available",
            "rows": 0,
            "countries": 0,
            "min_window_start": np.nan,
            "max_window_end": np.nan,
            "baseline_year": np.nan,
            "leakage_flags": np.nan,
        }
    ])

save_table(
    snapshot_volatility_leakage_check,
    "snapshot_volatility_leakage_check.csv",
    "Snapshot volatility leakage check",
    "Checks that diagnostic volatility windows do not include post-baseline years.",
)

display(snapshot_volatility_leakage_check)


Saved: snapshot_volatility_leakage_check.csv


,shock_id,rows,countries,min_window_start,max_window_end,baseline_year,leakage_flags
0,2007,44,44,2002,2007,2007,0
1,2019,44,44,2012,2019,2019,0


In [13]:

# ------------------------------------------------------
# DATA DICTIONARY
# ------------------------------------------------------

data_dictionary_rows = []

base_columns = master_country_year_panel.columns.tolist()

for col in base_columns:
    if col == "Country":
        description = "ISO3-style country code."
        role = "identifier"
    elif col == "Year":
        description = "Calendar year."
        role = "time_identifier"
    elif col in FINAL_5_RAW_VARIABLES:
        meta = FINAL_5_RAW_TO_SCORE[col]
        description = f"Raw input for {meta['concept']}. {meta['report_note']}"
        role = "final_ordering_raw_input"
    elif col in FINAL_5_SCORE_COLUMNS:
        description = "Direction-aligned 0–100 final ordering score."
        role = "final_ordering_score"
    elif col.startswith("Recovery_"):
        description = "GDP recovery outcome. Used only for validation, not for ordering."
        role = "validation_outcome"
    elif col.startswith("wgi_") or col.startswith("governance_"):
        description = "WGI governance context/sensitivity field. Not part of final POSet ordering."
        role = "context_or_sensitivity"
    else:
        description = "Master panel variable."
        role = "supporting_or_validation_variable"

    data_dictionary_rows.append({
        "column": col,
        "role": role,
        "description": description,
    })

# Add snapshot-only columns too.
for col in structural_snapshot_final5_2007_2019.columns:
    if col not in base_columns:
        if col in FINAL_5_SCORE_COLUMNS:
            role = "final_ordering_score"
            description = "Direction-aligned 0–100 final ordering score."
        elif col in {"shock_id", "shock_label", "baseline_year", "poset_temporal_design"}:
            role = "shock_snapshot_identifier"
            description = "Identifies the shock-specific baseline cross-section used for POSet."
        elif col.startswith("gdp_growth_stability") or col.startswith("gdp_growth_volatility"):
            role = "diagnostic_or_sensitivity_only"
            description = "Pre-shock GDP volatility/stability feature. Not final ordering."
        else:
            role = "snapshot_supporting_variable"
            description = "Structural snapshot support field."

        data_dictionary_rows.append({
            "column": col,
            "role": role,
            "description": description,
        })

master_country_year_panel_data_dictionary = (
    pd.DataFrame(data_dictionary_rows)
    .drop_duplicates("column")
    .sort_values(["role", "column"])
    .reset_index(drop=True)
)

save_table(
    master_country_year_panel_data_dictionary,
    "master_country_year_panel_data_dictionary.csv",
    "Master country-year panel data dictionary",
    "Column-level data dictionary for master panel and structural snapshots.",
)

display(master_country_year_panel_data_dictionary.head(100))


Saved: master_country_year_panel_data_dictionary.csv


,column,role,description
0,governance_context_role,context_or_sensitivity,WGI governance context/sensitivity field. Not ...
1,governance_context_score_0_100,context_or_sensitivity,WGI governance context/sensitivity field. Not ...
2,wgi_cc_score,context_or_sensitivity,WGI governance context/sensitivity field. Not ...
3,wgi_ge_score,context_or_sensitivity,WGI governance context/sensitivity field. Not ...
4,wgi_pv_score,context_or_sensitivity,WGI governance context/sensitivity field. Not ...
5,wgi_rl_score,context_or_sensitivity,WGI governance context/sensitivity field. Not ...
6,wgi_rq_score,context_or_sensitivity,WGI governance context/sensitivity field. Not ...
7,wgi_six_dimension_available_count,context_or_sensitivity,WGI governance context/sensitivity field. Not ...
8,wgi_six_dimension_mean_score,context_or_sensitivity,WGI governance context/sensitivity field. Not ...
9,wgi_va_score,context_or_sensitivity,WGI governance context/sensitivity field. Not ...


In [14]:

# ------------------------------------------------------
# REPORT-READY SCOPE AND MEASURE NOTES
# ------------------------------------------------------
# This directly addresses the professor/Francesco points around
# temporal scope, spatial scope, measurement length, and why 22-year data
# coverage does not imply a 22-year POSet profile.

report_ready_scope_and_measure_note = pd.DataFrame([
    {
        "topic": "Spatial scope",
        "report_text": (
            "The unit of analysis is the country. The empirical framework focuses on OECD-comparable "
            "country-level indicators, and the final POSet samples are determined by complete data availability "
            "for the five selected structural variables in the relevant baseline year."
        ),
    },
    {
        "topic": "Temporal scope",
        "report_text": (
            "The data acquisition and diagnostic panel covers 2002–2024/2025 depending on source availability. "
            "This longer panel is used for coverage checks, recovery construction, volatility diagnostics, "
            "and validation, not to define a single time-averaged POSet profile."
        ),
    },
    {
        "topic": "Length of measure",
        "report_text": (
            "Economic resilience is measured as a combination of pre-shock structural capacity and post-shock "
            "recovery behaviour. The structural capacity component is measured at the pre-shock baseline year, "
            "while recovery is measured over the subsequent years needed to return to the pre-shock GDP level."
        ),
    },
    {
        "topic": "POSet temporal design",
        "report_text": (
            "The POSet is constructed on two single cross-sections: 2007 for the Global Financial Crisis and "
            "2019 for the COVID shock. These years are chosen as representative pre-shock baselines because "
            "the research question concerns structural capacity immediately before major shocks."
        ),
    },
    {
        "topic": "Why not a 22-year POSet average",
        "report_text": (
            "A 22-year average would mix pre-shock and post-shock conditions, potentially contaminating the "
            "ordering variables with recovery outcomes. The long panel is therefore used for diagnostics and "
            "validation, while POSet ordering uses clean pre-shock snapshots."
        ),
    },
    {
        "topic": "Validation variables",
        "report_text": (
            "Validation variables are used to assess whether countries with stronger structural profiles also "
            "show more favourable post-shock macroeconomic behaviour. They are not used to construct the POSet "
            "ordering relation, which prevents outcome leakage."
        ),
    },
    {
        "topic": "Five final variables",
        "report_text": (
            "The final five ordering variables were chosen to represent distinct resilience-relevant capacities: "
            "fiscal capacity, labour-market strength, innovation capacity, human capital, and energy security. "
            "WGI governance and GDP volatility are retained only as contextual or sensitivity information."
        ),
    },
])

methodological_decision_updates_master = pd.DataFrame([
    {
        "decision": "Use two baseline cross-sections for POSet construction",
        "choice": "2007 and 2019",
        "reason": "They represent pre-shock structural conditions before the two validation shocks.",
        "risk_controlled": "Avoids mixing post-shock recovery outcomes into ordering variables.",
    },
    {
        "decision": "Keep final POSet ordering to five variables",
        "choice": ", ".join(FINAL_5_SCORE_COLUMNS),
        "reason": "Balances multidimensionality with interpretability and avoids governance redundancy.",
        "risk_controlled": "Reduces excessive dimensionality and avoids drifting into a single scalar index.",
    },
    {
        "decision": "Exclude WGI from main ordering",
        "choice": "Context/sensitivity only",
        "reason": "WGI dimensions are highly correlated and institutional context is better discussed separately.",
        "risk_controlled": "Avoids overloading the final POSet with correlated governance dimensions.",
    },
    {
        "decision": "Use recovery variables only for validation",
        "choice": "Recovery_2007 and Recovery_2019",
        "reason": "Recovery is an outcome, not a structural pre-shock capacity.",
        "risk_controlled": "Prevents leakage from validation outcome into the ordering variables.",
    },
])

save_table(
    report_ready_scope_and_measure_note,
    "report_ready_scope_and_measure_note.csv",
    "Report-ready scope and measurement note",
    "Report-ready paragraphs addressing temporal scope, spatial scope, length of measure, and POSet baseline design.",
)

save_table(
    methodological_decision_updates_master,
    "methodological_decision_updates_master.csv",
    "Methodological decision updates master",
    "Decision log entries for master dataset and final POSet design.",
)

display(report_ready_scope_and_measure_note)
display(methodological_decision_updates_master)


Saved: report_ready_scope_and_measure_note.csv
Saved: methodological_decision_updates_master.csv


,topic,report_text
0,Spatial scope,The unit of analysis is the country. The empir...
1,Temporal scope,The data acquisition and diagnostic panel cove...
2,Length of measure,Economic resilience is measured as a combinati...
3,POSet temporal design,The POSet is constructed on two single cross-s...
4,Why not a 22-year POSet average,A 22-year average would mix pre-shock and post...
5,Validation variables,Validation variables are used to assess whethe...
6,Five final variables,The final five ordering variables were chosen ...


,decision,choice,reason,risk_controlled
0,Use two baseline cross-sections for POSet cons...,2007 and 2019,They represent pre-shock structural conditions...,Avoids mixing post-shock recovery outcomes int...
1,Keep final POSet ordering to five variables,"debt_capacity_score_0_100, employment_strength...",Balances multidimensionality with interpretabi...,Reduces excessive dimensionality and avoids dr...
2,Exclude WGI from main ordering,Context/sensitivity only,WGI dimensions are highly correlated and insti...,Avoids overloading the final POSet with correl...
3,Use recovery variables only for validation,Recovery_2007 and Recovery_2019,"Recovery is an outcome, not a structural pre-s...",Prevents leakage from validation outcome into ...


In [15]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "06_Master_Dataset_Build_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    master_structural_snapshot_coverage.to_excel(writer, sheet_name="snapshot_coverage", index=False)
    structural_snapshot_final5_score_parameters.to_excel(writer, sheet_name="score_parameters", index=False)
    final5_ordering_variable_score_map.to_excel(writer, sheet_name="score_map", index=False)
    structural_snapshot_final5_complete_cases.to_excel(writer, sheet_name="complete_cases", index=False)
    master_problem_cases.to_excel(writer, sheet_name="problem_cases", index=False)
    snapshot_volatility_leakage_check.to_excel(writer, sheet_name="volatility_leakage", index=False)
    master_variable_coverage_summary.to_excel(writer, sheet_name="variable_coverage", index=False)
    master_country_year_panel_data_dictionary.to_excel(writer, sheet_name="data_dictionary", index=False)
    report_ready_scope_and_measure_note.to_excel(writer, sheet_name="scope_measure_note", index=False)
    methodological_decision_updates_master.to_excel(writer, sheet_name="decision_updates", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\06_Master_Dataset_Build_Audit.xlsx


In [16]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "master_dataset_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "master_dataset_table_inventory.csv", index=False)

expected_outputs = [
    "master_country_year_panel.csv",
    "master_country_year_panel_data_dictionary.csv",
    "master_variable_coverage_summary.csv",
    "structural_snapshot_final5_2007_2019.csv",
    "structural_snapshot_final5_complete_cases.csv",
    "structural_snapshot_final5_score_parameters.csv",
    "final5_ordering_variable_score_map.csv",
    "master_structural_snapshot_coverage.csv",
    "final5_score_range_summary.csv",
    "master_problem_cases.csv",
    "snapshot_volatility_leakage_check.csv",
    "report_ready_scope_and_measure_note.csv",
    "methodological_decision_updates_master.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("06 MASTER DATASET BUILD COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey outputs to inspect/send:")
print("- 06_Master_Dataset_Build_Audit.xlsx")
print("- master_structural_snapshot_coverage.csv")
print("- structural_snapshot_final5_score_parameters.csv")
print("- master_problem_cases.csv")
print("- report_ready_scope_and_measure_note.csv")

print("\nNext notebook:")
print("07_Pre_POSet_EDA_Checks_2002_5Var.ipynb")


06 MASTER DATASET BUILD COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,master_country_year_panel.csv,True,True,True
1,master_country_year_panel_data_dictionary.csv,True,True,True
2,master_variable_coverage_summary.csv,True,True,True
3,structural_snapshot_final5_2007_2019.csv,True,True,True
4,structural_snapshot_final5_complete_cases.csv,True,True,True
5,structural_snapshot_final5_score_parameters.csv,True,True,True
6,final5_ordering_variable_score_map.csv,True,True,True
7,master_structural_snapshot_coverage.csv,True,True,True
8,final5_score_range_summary.csv,True,True,True
9,master_problem_cases.csv,True,True,True



Key outputs to inspect/send:
- 06_Master_Dataset_Build_Audit.xlsx
- master_structural_snapshot_coverage.csv
- structural_snapshot_final5_score_parameters.csv
- master_problem_cases.csv
- report_ready_scope_and_measure_note.csv

Next notebook:
07_Pre_POSet_EDA_Checks_2002_5Var.ipynb
